    creating_array_deal
        Изучение способов добавления на сервер MetaTrader 5 массива сделок

In [1]:
# Инициализация необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
import os, sys
import pandas as pd
import numpy as np
current_dir = os.getcwd()                                               # Определяем путь к текущему файлу и его родительской директории
parent_dir = os.path.dirname(current_dir)                               # Формируем путь к libraries_py
libraries_path = os.path.join(parent_dir, "libraries_py")               # Формируем путь к libraries_py
sys.path.append(libraries_path)                                         # Добавляем libraries_py в sys.path

print (f"Обращаемся к файлк [config_loader.py], в директории {libraries_path}"),
from config_loader import load_config, setup_libraries, import_dynamic_functions

directories = load_config()
libraries_path = setup_libraries(directories)
import_functions, print_import_function_info = import_dynamic_functions(libraries_path)

if import_functions and print_import_function_info:
    modules_to_import = {
                "yar_sed_general_lib":  [libraries_path, 
                                    #"pd_set_option",
                                    #"CSVLoader",
                                    #"move_column",
                                    #"save_data_log_work_file",
                                    #"save_int_list_data_log_work_file",
                                    "list_print"],
                        'mt5_api':[
                            libraries_path,
                                'manager_connect_with_control',
                                'admin_connect_with_control',
                                'getting_array_trading_instruments'
                                 ],
                        "sed_array_lib": [libraries_path,
                                          "np_set_printoptions",
                            #"array_to_dataframe_with_multiline_headers",
                                      "array_to_dataframe"
                                      ]}                    # ключи — названия модулей, значения — списки функций
    
    imported = import_functions(modules_to_import)
    print_import_function_info(modules_to_import, imported)

    # Получаем нужные переменные
    directory_data_temp_files = directories.get("directory_data_temp_files", None)
    directory_data_log_files = directories.get("directory_data_log_files", None)
    if not directory_data_temp_files:
        raise ValueError("❌ ERROR: directory_data_temp_files не найден в конфигурации!")
    
else: print("❌ Ошибка при импорте функций.")

Обращаемся к файлк [config_loader.py], в директории c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py
Рабочая директория проекта c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
📂 directory_data_temp_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\working_data_files
📂 directory_data_log_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\log_data_files
📂 directory_libraries_path: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py
✅ Каталог c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py успешно добавлен в sys.path

 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'yar_sed_general_lib' успешен: ['list_print']
Импорт из 'mt5_api' успешен: ['manager_connect_with_control', 'admin_connect_with_control', 'getting_array_trading_instruments']
Импорт из 'sed_array_lib' успешен: ['np_set_printoptions', 'array_to_dataframe']

 Импортированные функции и их параметры:
Ф

In [5]:
beginning_of_period = 307718788     # Начало периода в котором предполагается удаление истории
end_of_period       = 1759407988    # Конец периода в котором предполагается удаление истории 
#login_list = account_ids

In [ ]:
# Полная Зачистка истории Пользователей <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
account_ids = [242719]
imported["list_print"](account_ids, "Начат процесс удаления торговой истории по счетам")

admin = imported["admin_connect_with_control"](admin if 'admin' in locals() else None)
print("admin = ", admin)
if admin:
    login_list = account_ids #                                                                                  # Определяем список торговых счетов

    order_array = admin.OrderRequestByLoginsNumPy(login_list)
    first_elements_order = imported["np_set_printoptions"](" \n Массив ОРДЕРОВ к удалению", order_array, 0, 3)

    deal_array = admin.DealRequestByLoginsNumPy(login_list, beginning_of_period, end_of_period)
    first_elements_deal = imported["np_set_printoptions"](" \n Массив СДЕЛОК к удалению", deal_array, 0, 3)     # Получаем список СДЕЛОК

    # Получение списка ПОЗИЦИЙ к Удалению <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    position_array = admin.PositionRequestByLoginsNumPy(login_list)
    if position_array is not None and isinstance(position_array, np.ndarray):
        # Удаление поля 'TimeCreateMsc'
        fields = list(position_array.dtype.names)                       # Получаем список всех полей
        fields.remove('TimeCreateMsc')                                  # Удаляем НЕ нужное поле

        position_array_2 = position_array[fields]                       # Создаем новый массив только с нужными полями
        first_elements_position = imported["np_set_printoptions"](" \n Массив ПОЗИЦИЙ к удалению", position_array_2, 27, 3) # Получаем список ПОЗИЦИЙ

    else:
        print(f"\n Массив ПОЗИЦИЙ к удалению отсутствует; position_array = {position_array}")
        first_elements_position = None
    # >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

    del deal_array

ПРИМЕР ВЫВОДА